# Week 1 · Day 4 — Revenue Precision & Advanced Aggregation

**Gap targets today:**
- ❌ Fix: `SUM(o.total_amount)` after joining `order_items` → must use `SUM(oi.quantity * oi.unit_price)`
- ❌ Fix: `GROUP BY` including `c.customer_id` inside segment-level queries → breaks rollup

**You're solid on (no re-teaching):** INNER JOIN, LEFT JOIN + IS NULL, GROUP BY logic, HAVING syntax, CASE WHEN sorting, three-table capstone structure

---

**Business First Prompt:** *You're a data analyst at a retail e-commerce company. The finance team flagged that your revenue numbers differ from the accounting system. The root cause: some queries summed order-level totals after exploding rows with a join — double-counting or using the wrong grain. Today you fix that, and build airtight segment-level reporting.*



In [1]:
import sqlite3
import pandas as pd

conn = sqlite3.connect('week1_ecommerce.db')
print('Connected to week1_ecommerce.db')

Connected to week1_ecommerce.db


---
## Concept 1 — The Order-Level vs Item-Level Revenue Trap

When you join `orders` to `order_items`, each order row **fans out** — one row per item.  
If you then `SUM(o.total_amount)`, you sum the order total **once per item**, which inflates revenue.

| Approach | What it sums | Risk |
|---|---|---|
| `SUM(o.total_amount)` after joining order_items | Order total × number of items in that order | **Double-counts** |
| `SUM(oi.quantity * oi.unit_price)` | Each line item's actual value | ✅ Correct item grain |
| `COUNT(DISTINCT o.order_id)` | Unique orders, not inflated rows | ✅ Always safe |

**Rule:** Once `order_items` is in the FROM/JOIN chain, revenue = `SUM(oi.quantity * oi.unit_price)`.

**Drill 1** — Show total item-level revenue and distinct order count per customer.  
Join `orders` and `order_items`. Use `SUM(oi.quantity * oi.unit_price)` for revenue.  
Order by revenue descending.

In [2]:
q1 = """SELECT o.customer_id, 
        SUM(oi.quantity * oi.unit_price) AS total_revenue,
        COUNT(DISTINCT o.order_id) AS distinct_orders

FROM orders o JOIN order_items oi ON o.order_id = oi.order_id
WHERE o.status = 'completed'
GROUP BY o.customer_id
ORDER BY total_revenue DESC

"""
df = pd.read_sql_query(q1, conn)
display(df)

,customer_id,total_revenue,distinct_orders
0,10,1350.0,2
1,3,958.5,3
2,1,595.0,3
3,4,571.0,2
4,2,555.0,3
5,6,438.0,2
6,7,220.0,1
7,5,180.0,1
8,9,160.0,1
9,8,89.0,1


**Drill 2** — Prove the trap. Write TWO queries side by side for customer_id = 1:  
Query A: `SUM(o.total_amount)` after joining order_items  
Query B: `SUM(oi.quantity * oi.unit_price)`  
Do they match? (Hint: they shouldn't if total_amount wasn't set to match item math.)

In [2]:
q2a = """ 
SELECT 
        SUM(o.total_amount) AS total_revenue
FROM orders o JOIN order_items oi ON o.order_id = oi.order_id
WHERE customer_id = 1

"""
df = pd.read_sql_query(q2a, conn)
display(df)

,total_revenue
0,1965.0


In [4]:
q2b = """
SELECT  SUM(oi.quantity * oi.unit_price) AS total_revenue
FROM orders o JOIN order_items oi ON o.order_id = oi.order_id
WHERE customer_id = 1


"""
df = pd.read_sql_query(q2b, conn)
display(df)

,total_revenue
0,623.0


**Drill 3** — Revenue by product category. Join all three tables.  
Group by `oi.category`. Show category, total revenue (`SUM(oi.quantity * oi.unit_price)`), and total units sold (`SUM(oi.quantity)`).  
Order by revenue descending.

In [9]:
q3 = """ SELECT oi.category,
        SUM(oi.quantity * oi.unit_price) AS total_revenue,
        SUM(oi.quantity) AS total_units_sold
        FROM order_items oi JOIN orders o ON oi.order_id = o.order_id
        WHERE o.status = 'completed'
        GROUP BY oi.category
        ORDER BY total_revenue DESC

"""
df = pd.read_sql_query(q3, conn)
display(df)

,category,total_revenue,total_units_sold
0,Electronics,3084.0,19
1,Furniture,1515.0,5
2,Accessories,517.5,15


---
## Concept 2 — GROUP BY Grain Must Match Your Analysis Level

**The Day 3 bug:** Including `c.customer_id` in GROUP BY when you want segment-level totals creates one row *per customer* instead of one row *per segment*. The grouping breaks.

| Goal | Correct GROUP BY | Wrong GROUP BY |
|---|---|---|
| Revenue per **segment** | `GROUP BY c.segment` | `GROUP BY c.segment, c.customer_id` ❌ |
| Revenue per **customer** | `GROUP BY c.customer_id, c.customer_name` | `GROUP BY c.segment` ❌ |
| Revenue per **country + segment** | `GROUP BY c.country, c.segment` | adding `c.customer_id` ❌ |

**Mental check:** *"What is one row in my output?"* Every column in SELECT that isn't aggregated must appear in GROUP BY — and nothing extra.

**Drill 4** — Revenue and order count per customer segment.  
One row per segment. Do NOT include `customer_id` in GROUP BY.  
Columns: segment, total_revenue, order_count.

In [10]:
q4 = """ SELECT c.segment,
        SUM(oi.quantity * oi.unit_price) AS total_revenue,
        COUNT(DISTINCT o.order_id) AS orders_count
        FROM customers c JOIN orders o ON c.customer_id = o.customer_id
        JOIN order_items oi ON o.order_id = oi.order_id
        WHERE o.status = 'completed'
        GROUP BY c.segment
        ORDER BY total_revenue DESC

"""
df = pd.read_sql_query(q4, conn)
display(df)

,segment,total_revenue,orders_count
0,VIP,2516.0,7
1,Regular,2111.5,9
2,New,489.0,3


In [11]:
q_check = "SELECT DISTINCT segment FROM customers"
df = pd.read_sql_query(q_check, conn)
display(df)

,segment
0,VIP
1,Regular
2,New


**Drill 5** — Revenue per country AND segment combination.  
One row per unique country+segment pair. Use `SUM(oi.quantity * oi.unit_price)`.  
Order by country, then revenue descending.

In [20]:
q5 = """ SELECT DISTINCT(c.country),
        c.segment,
        SUM(oi.quantity * oi.unit_price) AS total_revenue
        FROM orders o JOIN customers c ON o.customer_id = c.customer_id JOIN order_items oi ON o.order_id = oi.order_id
        WHERE o.status = 'completed'
        GROUP BY c.country, c.segment
        ORDER BY c.country, total_revenue DESC

"""
df = pd.read_sql_query(q5, conn)
display(df)

,country,segment,total_revenue
0,AU,New,89.0
1,CA,Regular,993.0
2,MX,VIP,1350.0
3,UK,VIP,571.0
4,US,Regular,1118.5
5,US,VIP,595.0
6,US,New,400.0


**Drill 6** — Average order value per segment (item-level).  
Step 1: total item revenue per order (subquery or CTE).  
Step 2: average that per segment.  
Columns: segment, avg_order_value. Round to 2 decimal places.

In [30]:
q6 = """SELECT c.segment,
        ROUND(AVG(o.total_amount), 2) AS avg_order_value
        FROM customers c JOIN orders o ON c.customer_id = o.customer_id
        WHERE o.status= 'completed'
        GROUP BY c.segment
        ORDER BY avg_order_value DESC
"""
df = pd.read_sql_query(q6, conn)
display(df)

,segment,avg_order_value
0,VIP,555.71
1,Regular,438.94
2,New,251.67


---
## Concept 3 — HAVING with Item-Level Aggregates

HAVING filters on aggregated values — it works the same whether you're at order grain or item grain.  
The key: the aggregate in HAVING must use the same expression as in SELECT.

```sql
-- If SELECT has SUM(oi.quantity * oi.unit_price)
-- HAVING must also use SUM(oi.quantity * oi.unit_price)
HAVING SUM(oi.quantity * oi.unit_price) > 500
-- NOT: HAVING SUM(o.total_amount) > 500
```

**Drill 7** — Customers with total item-level revenue above $300. Show customer_name and revenue.  
Filter with HAVING on `SUM(oi.quantity * oi.unit_price)`.

In [33]:
q7 = """ SELECT c.name,
        SUM (oi.quantity * oi.unit_price) AS revenue
        FROM customers c JOIN orders o ON c.customer_id = o.customer_id 
        JOIN order_items oi ON o.order_id = oi.order_id
        WHERE o.status = 'completed'
        GROUP BY c.name
        HAVING SUM (oi.quantity * oi.unit_price) > 300
        ORDER BY revenue DESC

"""
df = pd.read_sql_query(q7, conn)
display(df)

,name,revenue
0,Carlos Mendez,1350.0
1,Sara Lopez,958.5
2,Alice Martin,595.0
3,James Wu,571.0
4,Bob Chen,555.0
5,Omar Hassan,438.0


**Drill 8** — Categories with more than 3 total units sold AND revenue above $200.  
Use two HAVING conditions joined with AND.

In [37]:
q8 = """ SELECT oi.category,
        SUM(oi.quantity * oi.unit_price) AS total_revenue,
        SUM(oi.quantity) AS total_units_sold
        FROM order_items oi JOIN orders o ON oi.order_id = o.order_id
        JOIN customers c ON o.customer_id = c.customer_id
        WHERE o.status = 'completed'
        GROUP BY oi.category
        HAVING SUM(oi.quantity * oi.unit_price) > 200 AND SUM(oi.quantity) > 3
        ORDER BY total_revenue DESC

"""
df = pd.read_sql_query(q8, conn)
display(df)

,category,total_revenue,total_units_sold
0,Electronics,3084.0,19
1,Furniture,1515.0,5
2,Accessories,517.5,15


**Drill 9** — Segments where average item-level order value exceeds $250.  
Use a subquery to get per-order revenue first, then group by segment and apply HAVING.

In [39]:
q9 = """ SELECT c.segment,
        ROUND(AVG(o.total_amount), 2) AS avg_order_value
        FROM customers c JOIN orders o ON c.customer_id = o.customer_id
        WHERE o.status = 'completed'
        GROUP BY segment
        HAVING ROUND(AVG(o.total_amount), 2) > 250
        ORDER BY avg_order_value DESC

"""
df = pd.read_sql_query(q9, conn)
display(df)

,segment,avg_order_value
0,VIP,555.71
1,Regular,438.94
2,New,251.67


---
## Concept 4 — Mixing COUNT(DISTINCT) with SUM Across Joins

After joining `order_items`, you have multiple rows per order.  
- `COUNT(o.order_id)` counts rows — **inflated**, wrong  
- `COUNT(DISTINCT o.order_id)` counts unique orders — **correct**  
- `COUNT(DISTINCT oi.item_id)` counts line items — valid if that's the intent

Always be explicit about *what* you're counting.

**Drill 10** — Per category: distinct order count, total units, total item revenue, and average unit price.  
Use `COUNT(DISTINCT o.order_id)` for orders. Round avg unit price to 2 decimal places.

In [46]:
q10 = """ SELECT oi.category,
        COUNT(DISTINCT o.order_id) AS orders_count,
        COUNT(DISTINCT oi.item_id) AS total_units,
        SUM(oi.quantity * oi.unit_price) AS total_item_revenue,
        ROUND(AVG(oi.unit_price), 2) AS avg_unit_price
        FROM order_items oi JOIN orders o ON oi.order_id = o.order_id
        JOIN customers c ON o.customer_id = c.customer_id
        WHERE o.status = 'completed'
        GROUP BY oi.category
        
"""
df = pd.read_sql_query(q10, conn)
display(df)

,category,orders_count,total_units,total_item_revenue,avg_unit_price
0,Accessories,10,11,517.5,39.77
1,Electronics,14,17,3084.0,177.88
2,Furniture,4,5,1515.0,303.00


**Drill 11** — Per customer: distinct orders, distinct categories purchased, and total revenue.  
Show customers who bought from at least 2 categories (HAVING). Include customer_name.

In [122]:
q11 = """ SELECT c.name,
        COUNT(DISTINCT o.order_id) AS distinct_orders,
        COUNT(DISTINCT oi.category) AS distinct_categories,
        SUM(o.total_amount) AS total_revenue
        FROM customers c JOIN orders o ON c.customer_id = o.customer_id
        JOIN order_items oi ON o.order_id = oi.order_id
        WHERE o.status = 'completed'
        GROUP BY c.name
        HAVING COUNT(DISTINCT oi.category) > 2
        ORDER BY total_revenue DESC


"""
df = pd.read_sql_query(q11, conn)
display(df)

,name,distinct_orders,distinct_categories,total_revenue
0,Sara Lopez,3,3,2951.5
1,Alice Martin,3,3,1840.0
2,Bob Chen,3,3,1695.0
3,James Wu,2,3,1580.0


---
## Concept 5 — Subqueries for Grain Separation

Sometimes you need to aggregate at two different grains in one query.  
The pattern: compute item-level totals in a subquery, then group the subquery results by a higher grain.

```sql
SELECT segment, AVG(order_revenue) AS avg_order_value
FROM (
    SELECT c.segment,
           o.order_id,
           SUM(oi.quantity * oi.unit_price) AS order_revenue
    FROM customers c
    JOIN orders o ON c.customer_id = o.customer_id
    JOIN order_items oi ON o.order_id = oi.order_id
    GROUP BY c.segment, o.order_id   -- item → order grain
) sub
GROUP BY segment                      -- order → segment grain
```
Notice: `o.order_id` is in the inner GROUP BY but NOT the outer.

**Drill 12** — For each country: average revenue per order (item-level).  
Inner query: order-level revenue. Outer query: average by country.  
Round to 2 decimal places.

In [11]:
q12 = """SELECT country, ROUND(AVG(total_amount), 2) AS avg_revenue_per_order
        FROM ( SELECT SUM(oi.quantity * oi.unit_price) AS total_amount,
                c.country
                 FROM customers c JOIN orders o ON c.customer_id = o.customer_id JOIN order_items oi ON o.order_id = oi.order_id
                GROUP BY c.country, o.order_id
              ) sub
        GROUP BY country
        ORDER BY avg_revenue_per_order DESC

        """

df = pd.read_sql_query(q12, conn)
display(df)

,country,avg_revenue_per_order
0,MX,675.00
1,AU,284.50
2,UK,241.67
3,CA,193.71
4,US,180.03


**Drill 13** — Rank customers by total item-level revenue using a subquery.  
Inner query: revenue per customer. Outer query: add a revenue_rank column using `RANK()` or `ROW_NUMBER()`, or just ORDER BY with a row counter — your choice.  
Show top 5.

In [78]:
q13 = """ SELECT name, RANK() OVER(ORDER BY total_item_revenue DESC) AS revenue_rank
        FROM (SELECT c.name,
               SUM(oi.quantity * oi.unit_price) AS total_item_revenue
                FROM customers c JOIN orders o ON c.customer_id = o.customer_id JOIN order_items oi ON o.order_id = oi.order_id
                GROUP BY c.name
              ) sub
        GROUP BY name
        ORDER BY total_item_revenue DESC

"""
df = pd.read_sql_query(q13, conn)
display(df)

,name,revenue_rank
0,Carlos Mendez,1
1,Sara Lopez,2
2,Omar Hassan,3
3,James Wu,4
4,Alice Martin,5
5,Bob Chen,6
6,David Kim,7
7,Emma Rossi,8
8,Lily Thompson,9
9,Priya Nair,10


---
## Concept 6 — LEFT JOIN to Detect Gaps at Item Level

LEFT JOIN + IS NULL is still your tool for finding things that *don't* exist — but now at item grain.  
E.g., customers with no completed orders, orders with no items (data quality), products never ordered.

**Drill 14** — Customers who have NO completed orders (status = 'completed').  
Use LEFT JOIN + IS NULL on a filtered join or subquery. Show customer_name and segment.

In [83]:
q14 = """SELECT c.name,
        COUNT(DISTINCT o.order_id) AS orders_count
        FROM customers c LEFT JOIN orders o ON c.customer_id = o.customer_id
        WHERE status <> 'completed' OR status IS NULL
        GROUP BY c.name
        ORDER BY orders_count DESC


"""
df = pd.read_sql_query(q14, conn)
display(df)

,name,orders_count
0,Priya Nair,2
1,Lily Thompson,2
2,Sara Lopez,1
3,Omar Hassan,1
4,James Wu,1
5,Emma Rossi,1
6,David Kim,1
7,Bob Chen,1
8,Alice Martin,1


**Drill 15** — Find any orders that have no items in order_items (data quality check).  
LEFT JOIN order_items onto orders. Show order_id and order_date for orphans.

In [87]:
q15 = """ SELECT o.order_id,
            o.order_date
            FROM orders o LEFT JOIN order_items oi ON o.order_id = oi.order_id
            WHERE item_id IS NULL
            ORDER BY o.order_id

            
            
                 

"""
df = pd.read_sql_query(q15, conn)
display(df)

,order_id,order_date


---
## Capstone — Revenue Precision Audit Dashboard

**Scenario:** Finance wants a segment performance report. Previous analyst used `SUM(o.total_amount)` after joining order_items. Your job: build the correct version, flag the error, and add KPIs they didn't have.

Build ONE query with all of the following in a single result set:

| Column | Definition |
|---|---|
| segment | Customer segment |
| customer_count | Distinct customers |
| completed_orders | COUNT DISTINCT of completed orders only |
| item_revenue | `SUM(oi.quantity * oi.unit_price)` for completed orders only |
| avg_order_value | item_revenue / completed_orders, rounded to 2 decimal places |
| top_category | The category with highest revenue in that segment (use a correlated subquery or subquery join) |

Order by item_revenue descending.  
Filter: completed orders only (apply in JOIN or WHERE, not HAVING).

In [3]:
q_capstone = """
SELECT c.segment,
       COUNT(DISTINCT c.customer_id) AS customer_count,
       COUNT(DISTINCT o.order_id) AS completed_orders,
       SUM(oi.quantity * oi.unit_price) AS item_revenue,
       ROUND(SUM(oi.quantity * oi.unit_price) / COUNT(DISTINCT o.order_id), 2) AS avg_order_value,
       (SELECT oi2.category
        FROM order_items oi2
        JOIN orders o2 ON oi2.order_id = o2.order_id
        JOIN customers c2 ON o2.customer_id = c2.customer_id
        WHERE c2.segment = c.segment
          AND o2.status = 'completed'
        GROUP BY oi2.category
        ORDER BY SUM(oi2.quantity * oi2.unit_price) DESC
        LIMIT 1
       ) AS top_category
FROM customers c
JOIN orders o ON c.customer_id = o.customer_id
JOIN order_items oi ON o.order_id = oi.order_id
WHERE o.status = 'completed'
GROUP BY c.segment
ORDER BY item_revenue DESC
"""

df = pd.read_sql_query(q_capstone, conn)
display(df)

,segment,customer_count,completed_orders,item_revenue,avg_order_value,top_category
0,VIP,3,7,2516.0,359.43,Electronics
1,Regular,4,9,2111.5,234.61,Electronics
2,New,3,3,489.0,163.00,Electronics


---
## Day 4 Wrap-Up Checklist

Before submitting, confirm:

- [ ] Every revenue column uses `SUM(oi.quantity * oi.unit_price)` when order_items is in the join chain
- [ ] No `SUM(o.total_amount)` appears in any query that joins order_items
- [ ] GROUP BY grain matches the analysis level — no stray `customer_id` in segment queries
- [ ] `COUNT(DISTINCT o.order_id)` used wherever order count appears in a multi-item join
- [ ] Capstone: completed orders filtered before aggregation, not after